In [5]:
import sys
!{sys.executable} -m pip install psycopg2-binary python-dotenv

import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [6]:
# The connection string lives in a .env file at the project root (DATABASE_URL=postgresql://...).
load_dotenv(Path.cwd().parent / ".env")
Database_url = os.environ["DATABASE_URL"]

In [7]:
# Create the connection
# pool_pre_ping + pool_recycle stop Neon's serverless idle timeouts from leaving stale connections
engine = create_engine(
    Database_url,
    pool_pre_ping=True,
    pool_recycle=300,
    connect_args={"connect_timeout": 10},
)

# Test the connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database response:", result.fetchone())
        print("Successful connection")
except Exception as e:
    print("Connection failed:", repr(e))

Database response: (1,)
Successful connection


In [8]:
import os
from sqlalchemy import text

os.makedirs("../data/raw", exist_ok=True)

with engine.connect() as conn:
    for table in ["reviews", "bookings", "guests", "hotels"]:
        df = pd.read_sql(text(f"SELECT * FROM {table}"), conn)
        df.to_csv(f"../data/raw/{table}.csv", index=False)
        print(f"{table}: {df.shape}")

reviews: (35000, 8)
bookings: (35000, 9)
guests: (12000, 5)
hotels: (25, 5)


In [9]:
# Write the query
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""
table = pd.read_sql(query, engine)
table.head()

,table_name
0,bookings
1,guests
2,hotels
3,reviews
4,clean_bookings


In [10]:
# Load the booking data
booking = pd.read_sql("SELECT * FROM bookings", engine)
booking.head()

,booking_id,guest_id,hotel_id,room_type,booking_channel,check_in_date,check_out_date,total_amount,stay_duration
0,1,6699,18,Suite,Direct Website,2025-05-25,2025-05-26,296.44,1
1,2,11380,23,Deluxe,Corporate Portal,2025-01-09,2025-01-13,483.56,4
2,3,4812,13,Executive,Travel Agent,2025-03-04,2025-03-06,415.59,2
3,4,982,12,Family Room,Airbnb,2026-03-06,2026-03-09,601.18,3
4,5,615,5,Deluxe,Expedia,2025-02-25,2025-03-02,603.85,5


In [11]:
# Load the guests data
guests = pd.read_sql("SELECT * FROM guests", engine)
guests.head()

,guest_id,customer_type,loyalty_status,country,previous_stays
0,1,Leisure,None,Australia,0
1,2,Leisure,None,Germany,1
2,3,Business,Gold,United Kingdom,0
3,4,Family,Silver,United Arab Emirates,8
4,5,Corporate,None,France,0


In [12]:
guests.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   guest_id        12000 non-null  int64 
 1   customer_type   12000 non-null  object
 2   loyalty_status  5964 non-null   object
 3   country         12000 non-null  object
 4   previous_stays  12000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 468.9+ KB


In [13]:
booking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   booking_id       35000 non-null  int64  
 1   guest_id         35000 non-null  int64  
 2   hotel_id         35000 non-null  int64  
 3   room_type        35000 non-null  object 
 4   booking_channel  35000 non-null  object 
 5   check_in_date    35000 non-null  object 
 6   check_out_date   35000 non-null  object 
 7   total_amount     35000 non-null  float64
 8   stay_duration    35000 non-null  int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 2.4+ MB


# EDA

In [14]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots
from plotly.subplots import make_subplots    

In [15]:
## Total counts of room_type
room_type = booking['room_type'].value_counts().reset_index()
room_type

,room_type,count
0,Standard,10596
1,Deluxe,8619
2,Executive,5293
3,Family Room,4137
4,Suite,3544
5,Twin,2811


In [16]:
# Revenue by room_type
room_revenue = booking.groupby('room_type')['total_amount'].sum().reset_index()
room_revenue

,room_type,total_amount
0,Deluxe,3938732.66
1,Executive,3211430.94
2,Family Room,2158312.62
3,Standard,3112788.77
4,Suite,3029593.14
5,Twin,861141.58


In [17]:
# Booking channels
channels = booking['booking_channel'].value_counts().reset_index()
channels

,booking_channel,count
0,Direct Website,8688
1,Booking.com,7808
2,Expedia,5273
3,Travel Agent,3529
4,Airbnb,3416
5,Corporate Portal,3411
6,Walk-in,2875


In [18]:
# Booking channels
channel_revenue = booking.groupby('booking_channel')['total_amount'].sum().reset_index()
channel_revenue

,booking_channel,total_amount
0,Airbnb,1604609.36
1,Booking.com,3654237.93
2,Corporate Portal,1611396.00
3,Direct Website,4051385.20
4,Expedia,2452361.55
5,Travel Agent,1618571.44
6,Walk-in,1319438.23


In [19]:
print(booking['stay_duration'].describe())
print(booking['stay_duration'].value_counts().sort_index())

count    35000.000000
mean         2.954514
std          1.503233
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          7.000000
Name: stay_duration, dtype: float64
stay_duration
1     5299
2    10425
3     8679
4     5243
5     3614
7     1740
Name: count, dtype: int64


In [20]:
# Count bookings for each stay duration
stay = (booking['stay_duration'].value_counts().reset_index(name='count').sort_values('stay_duration'))
display(stay)

,stay_duration,count
2,1,5299
0,2,10425
1,3,8679
3,4,5243
4,5,3614
5,7,1740


In [21]:
# Create suplot
fig = make_subplots(rows=3, cols=2, subplot_titles=("Total Room Distribution", "Revenue Generated by Room Type", "Booking Channels Distribution", "Revenue Generated by Booking Channels",  "Stay Duration Distribution"))

# Add charts
fig.add_trace(go.Bar(x=room_type['room_type'], y=room_type['count'], name='Counts', marker_color='teal'), row=1, col=1)
fig.add_trace(go.Bar(x=room_revenue['room_type'], y=room_revenue['total_amount'], name='Revenue', marker_color='indianred'), row=1, col=2)
fig.add_trace(go.Bar(x=channels['booking_channel'], y=channels['count'], name='Counts', marker_color='blue'), row=2, col=1)
fig.add_trace(go.Bar(x=channel_revenue['booking_channel'], y=channel_revenue['total_amount'], name='Revenue', marker_color='orange'), row=2, col=2)
fig.add_trace(go.Bar(x=stay['stay_duration'], y=stay['count'], name='Counts', marker_color='purple'), row=3, col=1)

fig.update_layout(height=800, width=1200, title_text="Booking Data Analysis", showlegend=False)
fig.show()